# ChurnLens — 04. Churn Prediction & Explainability Engine
## Machine Learning Benchmarks, Threshold Optimization & SHAP Explainability

### Analytical Workflow:
1. Feature Preprocessing (Numerical scaling, categorical one-hot encoding)
2. Stratified Train/Test Split (80/20)
3. Model Benchmarking:
   - **Logistic Regression** (Interpretable baseline)
   - **Random Forest** (Tree ensemble)
   - **Gradient Boosting** (High-capacity boosting)
4. Evaluation: ROC-AUC, PR-AUC, Confusion Matrix, Brier Calibration Score
5. **Business Cost-Benefit Threshold Optimization**:
   - Maximizing retained revenue minus false-alarm intervention cost
6. Feature Importances and SHAP-aligned Drivers


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, auc, 
    confusion_matrix, classification_report, brier_score_loss
)

df = pd.read_csv('../data/cleaned_churn_data.csv')


### 1. Feature Preparation & Train/Test Partition


In [ ]:
feature_cols = [
    'age', 'customer_segment', 'subscription_plan', 'monthly_spend',
    'acquisition_channel', 'payment_method', 'tenure_months',
    'plan_changes', 'downgrades', 'days_since_last_login',
    'sessions_last_30_days', 'average_session_duration',
    'feature_usage_count', 'key_feature_usage', 'activity_change_pct',
    'support_tickets', 'unresolved_tickets', 'complaints', 'failed_payments'
]

cat_cols = ['customer_segment', 'subscription_plan', 'acquisition_channel', 'payment_method']
num_cols = [c for c in feature_cols if c not in cat_cols]

X = df[feature_cols]
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {len(X_train):,} | Testing set: {len(X_test):,}")


### 2. Model Training & Comparison


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42)
}

fitted_models = {}
eval_results = []

for name, clf in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    
    roc = roc_auc_score(y_test, y_prob)
    p, r, _ = precision_recall_curve(y_test, y_prob)
    pr = auc(r, p)
    brier = brier_score_loss(y_test, y_prob)
    
    eval_results.append({
        'Model': name,
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr, 4),
        'Brier Score': round(brier, 4)
    })

pd.DataFrame(eval_results)


### 3. ROC & Precision-Recall Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, pipe in fitted_models.items():
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, y_prob):.3f})")
    
    p, r, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(r, p, label=f"{name} (PR-AUC = {auc(r, p):.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=11, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

axes[1].set_title('Precision-Recall Curve', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()


### 4. Business Cost-Benefit Threshold Optimization
Finding the optimal decision threshold that maximizes net retained customer value.


In [ ]:
best_pipe = fitted_models['Gradient Boosting']
best_probs = best_pipe.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
cost_curves = []

for th in thresholds:
    preds = (best_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    # Net economic value calculation
    # True Positive retained value: $464 | FP intervention cost: -$40 | FN unaddressed churn: -$1440
    net_val = (tp * 464) - (fp * 40) - (fn * 1440)
    cost_curves.append({'threshold': th, 'net_value': net_val, 'tp': tp, 'fp': fp, 'fn': fn})

cost_df = pd.DataFrame(cost_curves)
opt = cost_df.loc[cost_df['net_value'].idxmax()]

plt.figure(figsize=(10, 4.5))
plt.plot(cost_df['threshold'], cost_df['net_value'] / 1000, color='#1b9e77', linewidth=2.5)
plt.axvline(opt['threshold'], color='red', linestyle='--', label=f"Optimal Threshold: {opt['threshold']:.2f}")
plt.title('Net Business Value Created vs Decision Threshold ($ in Thousands)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Prediction Probability Threshold')
plt.ylabel('Net Value Generated ($K)')
plt.legend()
plt.tight_layout()
plt.show()


### 5. Feature Importance & Drivers


In [ ]:
clf = best_pipe.named_steps['classifier']
enc = best_pipe.named_steps['preprocessor'].named_transformers_['cat']
cat_names = enc.get_feature_names_out(cat_cols).tolist()
all_names = num_cols + cat_names

feat_imp = pd.DataFrame({
    'Feature': all_names,
    'Importance': clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp.head(10), x='Importance', y='Feature', palette='crest')
plt.title('Top 10 Feature Importances (Gradient Boosting Model)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()
